# Modelo 1 · Regresión: precio de viviendas en California

**Objetivo:** predecir un valor **numérico continuo** (el valor mediano de las viviendas de un sector) a partir de sus características.

**Dataset:** *California Housing* (censo de EE.UU. de 1990), la versión del libro *Hands-On Machine Learning*. Se carga directo desde GitHub, sin cuenta de Kaggle.

| Columna | Descripción |
|---|---|
| `longitude`, `latitude` | Ubicación del sector |
| `housing_median_age` | Antigüedad mediana de las casas |
| `total_rooms`, `total_bedrooms` | Total de habitaciones y dormitorios del sector |
| `population`, `households` | Población y número de hogares |
| `median_income` | Ingreso mediano (en decenas de miles de USD) |
| `ocean_proximity` | Cercanía al océano (categórica) |
| **`median_house_value`** | **Objetivo:** valor mediano de la vivienda (USD) |

**Flujo:** cargar → explorar → separar train/test → preprocesar → entrenar → evaluar → interpretar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Cargar los datos

In [ ]:
URL = "https://raw.githubusercontent.com/ageron/data/main/housing/housing.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

## 2. Exploración (EDA)

Revisamos tipos, valores faltantes y distribuciones antes de modelar.

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
df["ocean_proximity"].value_counts()

In [ ]:
df.hist(bins=40, figsize=(14, 9))
plt.tight_layout()
plt.show()

> **Ojo:** `median_house_value` tiene un tope en 500.001 USD (los valores fueron recortados en el censo) y `total_bedrooms` tiene valores faltantes. Los datos reales tienen "trampas".

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlación entre variables")
plt.show()

In [ ]:
df.plot.scatter(x="longitude", y="latitude", c="median_house_value", cmap="viridis",
                s=df["population"] / 100, alpha=0.4, figsize=(9, 7))
plt.title("Valor de la vivienda según ubicación")
plt.show()

## 3. Separar en entrenamiento y prueba

El modelo **aprende** con `train` y lo **evaluamos** con `test`, datos que nunca vio.

In [ ]:
X = df.drop(columns="median_house_value")
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 4. Preprocesamiento

- **Numéricas:** rellenar faltantes (`total_bedrooms`) con la mediana y estandarizar.
- **Categórica** (`ocean_proximity`): *one-hot encoding*.

Va dentro de un `Pipeline` para que se aprenda **solo con train** (evita *data leakage*).

In [ ]:
numericas = X.select_dtypes("number").columns.tolist()
categoricas = ["ocean_proximity"]

preprocesamiento = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
])

## 5. Entrenar modelos

Comparamos un modelo simple (**regresión lineal**) con uno más flexible (**Random Forest**).

In [ ]:
modelos = {
    "Regresión lineal": Pipeline([
        ("prep", preprocesamiento),
        ("modelo", LinearRegression()),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocesamiento),
        ("modelo", RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

predicciones = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    predicciones[nombre] = modelo.predict(X_test)

## 6. Evaluar

- **MAE**: error absoluto promedio (en USD, las mismas unidades que `y`).
- **RMSE**: castiga más los errores grandes.
- **R²**: proporción de la varianza explicada (1 = perfecto, 0 = igual que predecir el promedio).

In [ ]:
resultados = pd.DataFrame({
    nombre: {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R²": r2_score(y_test, pred),
    }
    for nombre, pred in predicciones.items()
}).T.round(3)
resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (nombre, pred) in zip(axes, predicciones.items()):
    ax.scatter(y_test, pred, alpha=0.2, s=8)
    ax.plot([0, 500_000], [0, 500_000], "r--", label="Predicción perfecta")
    ax.set(title=nombre, xlabel="Valor real (USD)", ylabel="Valor predicho (USD)")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Interpretar

In [ ]:
lineal = modelos["Regresión lineal"]
nombres = lineal.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lineal.named_steps["modelo"].coef_, index=nombres).sort_values()
coefs.plot.barh(title="Coeficientes de la regresión lineal (variables estandarizadas)")
plt.show()

In [ ]:
rf = modelos["Random Forest"]
importancias = pd.Series(rf.named_steps["modelo"].feature_importances_, index=nombres).sort_values()
importancias.plot.barh(title="Importancia de variables (Random Forest)")
plt.show()

## Conclusiones

- El ingreso del sector (`median_income`) es la variable más predictiva.
- Random Forest captura relaciones **no lineales** (por ejemplo, la ubicación) y logra un R² bastante mejor que la regresión lineal.
- La regresión lineal es más fácil de **explicar**: cada coeficiente dice cuánto sube o baja el precio.

## Ejercicios

1. Elimina `latitude`, `longitude` y `ocean_proximity` y vuelve a entrenar. ¿Cuánto empeora el R² de cada modelo?
2. Crea variables nuevas: `rooms_per_household = total_rooms / households` y `bedrooms_ratio = total_bedrooms / total_rooms`. ¿Mejora el modelo lineal?
3. Prueba distintos valores de `n_estimators` y `max_depth` en el Random Forest. ¿Ves overfitting? (Compara el R² en train y en test.)
4. Elimina las filas con `median_house_value >= 500_000` (el tope). ¿Cambian las métricas?